# Try Fintan for free

Fintan is an open model that keeps learning after release. This notebook runs **Fintan-4B** on a free GPU (Colab or
Kaggle, T4 with 16 GB): you ask it about the week of news it already learnt, teach it something new, let it learn,
and ask again.

Runtime: **GPU (T4)**. The first start downloads about 10 GB and takes a few minutes; a learning run takes longer on a
T4 than on a newer card. Everything stays in this notebook's machine; nothing you teach it is sent anywhere.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# the libraries the release was tested with
!pip install -q "transformers==5.24.1" "peft==0.22.0" accelerate "bitsandbytes==0.50.2" huggingface_hub

In [ ]:
import os, subprocess, time, json, urllib.request
from huggingface_hub import snapshot_download
MODEL = "fintanlabs/Fintan-4B"
APP = "/content/fintan" if os.path.isdir("/content") else "/kaggle/working/fintan"
snapshot_download(MODEL, local_dir=APP)
print("downloaded to", APP)

In [ ]:
# start Fintan: a 4-bit engine (fits the T4) and the front door at http://127.0.0.1:8900
env = dict(os.environ, FINTAN_COMPUTE_DTYPE="float16")      # the T4 has no bfloat16
server = subprocess.Popen(["bash", "scripts/serve_small.sh"], cwd=APP, env=env,
                          stdout=open("/tmp/fintan.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8900/fintan/status", timeout=5); print("Fintan is ready"); break
    except Exception:
        time.sleep(10)

def call(path, body=None):
    req = urllib.request.Request("http://127.0.0.1:8900" + path, method="POST" if body is not None else "GET",
                                 data=json.dumps(body).encode() if body is not None else None,
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=900).read())

def ask(q):
    r = call("/v1/chat/completions", {"model": "fintan", "messages": [{"role": "user", "content": q}],
                                       "max_tokens": 200, "chat_template_kwargs": {"enable_thinking": False}})
    return r["choices"][0]["message"]["content"]

## 1. It already knows a week of news (18 to 23 November 2026)
Plain Qwen3.5-4B has never seen this week.

In [ ]:
print(ask("Who is the President of Ukraine as of November 23, 2026?"))
print(ask("Who am I talking to?"))

## 2. Teach it something
Anything in text: notes, a document, facts about you. Or just tell it in chat ("remember that ...").

In [ ]:
call("/fintan/settings", {"auto_learn": False})     # we start the run ourselves below
r = call("/fintan/teach", {"kind": "documents", "source": "notebook",
    "text": "The Lindgrove Rowing Club was founded in 1911. Its boathouse is on the River Avon, and members train "
            "every Saturday at 7 in the morning. The club's oldest boat, the Heron, was built in 1923."})
print(r)
print("Before learning:", ask("When do members of the Lindgrove Rowing Club train?"))

## 3. Let it learn
While it learns, the model pauses answering. On a T4 this takes a while.

In [ ]:
print(call("/fintan/learn", {}))
t0 = time.time()
while True:
    time.sleep(30)
    lr = call("/fintan/status")["learning"]
    if not lr["running"]:
        break
    print("learning... %d min" % ((time.time() - t0) / 60))
print(lr["last"])

In [ ]:
print("After learning:", ask("When do members of the Lindgrove Rowing Club train?"))
print(ask("When was the Lindgrove Rowing Club founded, and what is its oldest boat?"))

## Next
- Run it on your own GPU, with Docker and a chat app: https://github.com/troybrandonc-bit/fintan
- The models, with every result (failures included): https://huggingface.co/fintanlabs